# Near-duplicate discovery with Stipple

**Recipe:** render an embedding of your dataset, look for tight micro-clusters in the projection, lasso them to recover the row indices of suspected near-duplicates.

This example uses **synthetic** 2D embeddings with planted near-duplicate clusters so the visualization is reproducible. In real use you'd substitute MinHash-LSH projections, sentence-embedding projections, etc.

## Setup
200k synthetic 'documents' embedded in 2D. 195k are diffuse Gaussian noise; 5k form 50 tight planted near-duplicate clusters of size ~100, scattered across the embedding space.

In [1]:
import numpy as np
from stipple import Stipple

In [2]:
rng = np.random.default_rng(0)
n_noise = 195_000
n_clusters = 50
per_cluster = 100
n_dups = n_clusters * per_cluster
n = n_noise + n_dups

# Diffuse 'unique' documents (background)
x_noise = rng.standard_normal(n_noise).astype(np.float32) * 3.0
y_noise = rng.standard_normal(n_noise).astype(np.float32) * 3.0

# Planted near-duplicate clusters: 50 cluster centers in the same range,
# each producing ~100 points within a tight (~0.05) Gaussian.
centers = rng.standard_normal((n_clusters, 2)).astype(np.float32) * 3.0
x_dups = np.empty(n_dups, dtype=np.float32)
y_dups = np.empty(n_dups, dtype=np.float32)
for k in range(n_clusters):
    sl = slice(k * per_cluster, (k + 1) * per_cluster)
    x_dups[sl] = centers[k, 0] + rng.standard_normal(per_cluster).astype(np.float32) * 0.04
    y_dups[sl] = centers[k, 1] + rng.standard_normal(per_cluster).astype(np.float32) * 0.04

x = np.concatenate([x_noise, x_dups])
y = np.concatenate([y_noise, y_dups])
labels = np.concatenate([
    np.full(n_noise, 'noise'),
    np.full(n_dups, 'planted-dup'),
])

print(f'{n:,} total points · {n_noise:,} background · {n_dups:,} planted duplicates in {n_clusters} clusters')
print(f'Planted-dup indices: [{n_noise:,}, {n:,})')

200,000 total points · 195,000 background · 5,000 planted duplicates in 50 clusters
Planted-dup indices: [195,000, 200,000)


In [3]:
w = Stipple(x=x, y=y, color=labels)
w

**Try:** zoom into a high-density spot (wheel up to zoom). The planted duplicate clusters appear as tight orange specks against the blue background haze. Shift+drag around one to lasso it.

In [4]:
idx = w.selected_indices
if len(idx) == 0:
    print('No selection yet. Zoom into a tight cluster and shift+drag to lasso it.')
else:
    planted = (idx >= n_noise).sum()
    purity = planted / max(1, len(idx))
    print(f'Selected {len(idx):,} points · {purity:.1%} from planted-dup region')
    if planted > 0:
        # Show which planted cluster(s) we hit
        local = idx[idx >= n_noise] - n_noise
        clusters = np.unique(local // per_cluster)
        print(f'Planted clusters hit: {clusters.tolist()}')

No selection yet. Zoom into a tight cluster and shift+drag to lasso it.
